In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Load Attendance

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format', 'csv')\
        .option('cloudFiles.schemaEvolutionMode', 'rescue')\
            .option('cloudFiles.schemaLocation','abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/attendance/')\
                .load('abfss://source@myprojectssa.dfs.core.windows.net/attendance/')

In [0]:
df.writeStream.format('delta')\
    .option('checkpointLocation', 'abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/attendance/')\
        .trigger(once=True)\
            .option('path', 'abfss://bronze@myprojectssa.dfs.core.windows.net/attendance/')\
                .toTable('project_cata.bronze.attendance')

## Load Employee changes

In [0]:
df_emp_changes = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format', 'csv')\
        .option('cloudFiles.schemaLocation', 'abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/employee_changes/')\
            .option('cloudFiles.schemaEvolutionMode', 'rescue')\
                .load('abfss://source@myprojectssa.dfs.core.windows.net/employee_changes/')

In [0]:
df_emp_changes.writeStream.format('delta')\
                            .option('checkpointLocation', 'abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/employee_changes/')\
                                .trigger(once=True)\
                                    .option('path', 'abfss://bronze@myprojectssa.dfs.core.windows.net/employee_changes/')\
                                        .toTable('project_cata.bronze.employee_changes')

## Load Employees

In [0]:
df_emp = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','csv')\
        .option('cloudFiles.schemaLocation','abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/employees/')\
            .option('cloudFiles.schemaEvolutionMode', 'rescue')\
                .load('abfss://source@myprojectssa.dfs.core.windows.net/employees/')

In [0]:
df_emp.writeStream.format('delta')\
    .option('checkpointLocation', 'abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/employees/')\
        .trigger(once=True)\
            .option('path', 'abfss://bronze@myprojectssa.dfs.core.windows.net/employees/')\
                .toTable('project_cata.bronze.employees')

## Load Payroll

In [0]:
df_payroll = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format', 'csv')\
        .option('cloudFiles.schemaLocation', 'abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/payroll/')\
            .option('cloudFiles.schemaEvolutionMode', 'rescue')\
                .load('abfss://source@myprojectssa.dfs.core.windows.net/payroll/')

In [0]:
df_payroll.writeStream.format('delta')\
    .option('checkpointLocation', 'abfss://bronze@myprojectssa.dfs.core.windows.net/checkpoints/payroll/')\
        .trigger(once=True)\
            .option('path','abfss://bronze@myprojectssa.dfs.core.windows.net/payroll/')\
                .toTable('project_cata.bronze.payroll')

## Load XML Files

In [0]:
base_landing = "abfss://source@myprojectssa.dfs.core.windows.net/xml"
base_bronze  = "abfss://bronze@myprojectssa.dfs.core.windows.net"

xml_sources = [
    {"name": "departments", "row_tag": "department"},
    {"name": "job_roles",    "row_tag": "job_role"},
    {"name": "locations",   "row_tag": "location"},
]

for source in xml_sources:
    df_xml = spark.read.format('xml')\
        .option('rowTag', source['row_tag'])\
            .option('path', f"{base_landing}/{source['name']}.xml")\
                .load()

    df_xml.write.format('delta')\
        .mode('overwrite')\
        .option('path', f"{base_bronze}/{source['name']}/")\
            .saveAsTable(f"project_cata.bronze.{source['name']}")